# 🤖 03 - Model Training & Experiments
## NBE Credit Risk Intelligence

**Purpose:** Train and compare multiple models

**Author:** NBE Credit Risk Team | **Date:** February 2026

## 1️⃣ Setup

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported!')

## 2️⃣ Load Processed Data

In [ ]:
df = pd.read_csv('../data/processed/german_credit_fe_v3.csv')
X = df.drop('Risk', axis=1)
y = df['Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'✅ Data ready: Train={len(X_train)}, Test={len(X_test)}')

## 3️⃣ Train Multiple Models

In [ ]:
models = {
    'Logistic Regression (V1)': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Random Forest (V3)': RandomForestClassifier(
        n_estimators=100, max_depth=15, random_state=42,
        class_weight='balanced', n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, random_state=42
    )
}

results = {}

for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train_scaled))
    test_acc  = accuracy_score(y_test,  clf.predict(X_test_scaled))
    cm = confusion_matrix(y_test, clf.predict(X_test_scaled))
    results[name] = {
        'train_acc': train_acc,
        'test_acc':  test_acc,
        'fn_cases':  cm[1, 0]
    }
    print(f'{name}: Train={train_acc*100:.2f}% | Test={test_acc*100:.2f}% | FN={cm[1,0]}')

## 4️⃣ Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T
results_df.columns = ['Train Accuracy', 'Test Accuracy', 'False Negatives']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Accuracy comparison
x = range(len(results_df))
axes[0].bar([i-0.2 for i in x], results_df['Train Accuracy']*100,
           0.4, label='Train', color='#006341', alpha=0.8)
axes[0].bar([i+0.2 for i in x], results_df['Test Accuracy']*100,
           0.4, label='Test', color='#D4AF37', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df.index, rotation=15)
axes[0].set_title('Model Accuracy Comparison')
axes[0].legend()
axes[0].set_ylabel('Accuracy (%)')

# False Negatives comparison
axes[1].bar(results_df.index, results_df['False Negatives'],
           color='#e74c3c', alpha=0.8)
axes[1].set_title('False Negatives Comparison')
axes[1].set_xticklabels(results_df.index, rotation=15)
axes[1].set_ylabel('FN Cases')

plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(results_df)

## 5️⃣ Winner: Random Forest

In [ ]:
print('='*60)
print('🏆 WINNER: Random Forest Classifier')
print('='*60)
print('Reason: Highest test accuracy (76.5%)')
print('→ Next: model_training_final.ipynb (Production)')